# Workshop 4: Open-TYNDP Outcomes and CBA Workflows

:::{note} At the end of this notebook, you will be able to:

**1. Inspect Open-TYNDP benchmarks**
  - Navigate and analyze Open-TYNDP NT scenario outcomes using PyPSA's statistics module
  - Investigate latest networks using PyPSA-Explorer
  - Compare your findings with the benchmarking framework
  
**2. Run CBA workflows**
  - Modify the CBA configuration for the targeted project and/or climate year
  - Learn how to execute CBA analysis coupled to or detached from the Scenario Building workflow

:::

:::{note}
If you have not set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Click the rocket button in the top right corner and launch "Colab". If that doesn't work, download the `.ipynb` file and import it in [Google Colab](https://colab.research.google.com/).

Then install the required packages by uncommenting the cell below.

:::

In [ ]:
# uncomment for running this notebook on Colab
# !pip install pypsa==1.2.1 pypsa-explorer pandas matplotlib numpy pdf2image cartopy
# !apt-get install poppler-utils

In [ ]:
import os
from datetime import datetime
import pandas as pd
import pypsa
import shutil
import zipfile
from urllib.request import urlretrieve
from pdf2image import convert_from_path
from pdf2image.exceptions import PDFPageCountError
from IPython.display import SVG, Code, IFrame, Image, display
import matplotlib.pyplot as plt
from pypsa_explorer import create_app
from pathlib import Path
import cartopy.crs as ccrs
from pypsa.plot.maps.static import (
    add_legend_lines,
)

pypsa.set_option("params.statistics.nice_names", True)
pypsa.set_option("params.statistics.drop_zero", True)
pypsa.set_option("params.statistics.round", 3)
plt.rcParams["figure.figsize"] = [14, 7]
clip = 1  # TWh

In [ ]:
def unzip_with_timestamps(zip_path, extract_to, keep_zip=True):
    """Unzip a file while preserving original file timestamps."""
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            # Extract the file
            zip_ref.extract(member, extract_to)

            # Get the extracted file path
            extracted_path = os.path.join(extract_to, member.filename)

            # Get the modification time from the zip file
            date_time = datetime(*member.date_time)
            timestamp = date_time.timestamp()

            # Set both access and modification times
            os.utime(extracted_path, (timestamp, timestamp))
    if not keep_zip:
        os.remove(zip_path)

As in previous workshops, we first need to retrieve some files that we'll need today.

In [ ]:
urls = {
    "data/results-0.7.1.zip": "https://storage.googleapis.com/open-tyndp-data-store/outcomes/0.7.1/results-0.7.1.zip",
    "scripts/_helpers.py": "https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/feat/workshop-4/open-tyndp-workshops/scripts/_helpers.py",
}

os.makedirs("data", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
for name, url in urls.items():
    if os.path.exists(name):
        print(f"File {name} already exists. Skipping download.")
    else:
        print(f"Retrieving {name} from storage.")
        urlretrieve(url, name)
        print(f"File available in {name}.")

to_dir = "data/results-0.7.1"
if not os.path.exists(to_dir):
    print(f"Unzipping data/results-0.7.1.zip.")
    unzip_with_timestamps("data/results-0.7.1.zip", "data/results-0.7.1")
print(f"Latest NT results for Open-TYNDP v0.7.1 are available in '{to_dir}'.")

print("Done")

And we'll also import some handy helper functions that we introduced in the last workshops.

In [ ]:
from scripts._helpers import (
    show_benchmarks,
    display_code_lines,
    run_pypsa_explorer_in_colab,
)

# Scenario Building

## Explore NT outcomes using the ``PyPSA.statistics`` module

`n.statistics` provides a consistent, high-level API that handles component iteration, port mapping, and carrier grouping automatically.

:::{tip}

`n.stats` is available as a shorthand alias for `n.statistics`.

:::

Each method can be called individually or explored via the summary table:

| Category | Methods |
|---|---|
| **Costs** | `capex()`, `installed_capex()`, `expanded_capex()`, `opex()`, `system_cost()` |
| **Capacity** | `installed_capacity()`, `optimal_capacity()`, `expanded_capacity()`, `capacity_factor()` |
| **Energy** | `supply()`, `withdrawal()`, `energy_balance()`, `transmission()`, `curtailment()` |
| **Market** | `prices()`, `revenue()`, `market_value()` |

Every method accepts the same filtering and grouping parameters:
| Parameter | Description |
|---|---|
| `groupby` | String, list, or callable — how to group results (default: `\"carrier\"`) |
| `groupby_method` | Aggregation function (`\"sum\"` (default), `\"mean\"`, …) |
| `groupby_time` | `\"sum\"`, `\"mean\"`, or `False` for time series — default varies by method |
| `components` | Filter to specific component types |
| `carrier` | Filter by carrier name (internal name) |
| `bus_carrier` | Filter by the carrier of the bus |
| `nice_names` | Use human-readable carrier names (default: `True`) |

:::{warning}

`prices()` has a simplified interface — `groupby` and `groupby_time` are booleans,
and it does not accept `carrier` or `components`.

:::

The full `PyPSA.statistics` API documentation is available in the [pypsa documentation](https://docs.pypsa.org/latest/user-guide/statistics/). Additionally, you can find two video tutorials on *PyPSA meets Earth*'s youtube channel ([part 1](https://www.youtube.com/watch?v=_Asjhz6We5o), [part 2](https://www.youtube.com/watch?v=nlh3azf2JJw)) for more comprehensive information and examples on how to use the statistics module. This learning material is open-source and available on [GitHub](https://github.com/open-energy-transition/pypsa-training-sessions).

### Reminder: Extracting insights from the network

Let's load the latest Open-TYNDP NT scenario outcomes (v0.7.1) again and explore them using the statistics module.

In [ ]:
# Load latest NT scenario networks
base_path = "data/results-0.7.1/NT-cy2009-20260520/networks/"


def import_network(fn: str):
    n = pypsa.Network(fn)
    n.carriers.loc["none", "color"] = "#000000"
    return n


# Load networks directly into dictionary for PyPSA-Explorer
networks = {
    "NT 2030": import_network(base_path + "base_s_all___2030.nc"),
    "NT 2040": import_network(base_path + "base_s_all___2040.nc"),
}

To start of we'll have a look at the network for the `NT 2030` scenario.

In [ ]:
scenario = "NT 2030"


For convenience, let's save the statistics accessor in a variable `s`.

In [ ]:
s2030 = networks[scenario].stats

You can easily get a comprehensive overview of all system-level metrics at once.

In [ ]:
s2030()

Of course, this can be a bit difficult to grasp. So let's have a look at some specific metrics instead, for example installed renewable capacities:

In [ ]:
(
    s2030.installed_capacity(
        bus_carrier="AC|low voltage",
        comps="Generator",
    )
    .div(1e3)  # GW
    .round(3)
    .to_frame("Installed Capacity [GW]")
    .filter(regex="^(?!Demand).*", axis=0)
)

We can also easily look into the outputs of the model.

So, let's investigate electricity supply and demand for our NT 2030 network using the `energy_balance` method:

In [ ]:
balance = (
    s2030.energy_balance(
        bus_carrier=["AC", "low voltage"],
        comps=["Generator", "Link", "StorageUnit", "Load"],
        groupby=["carrier"],
        aggregate_across_components=True,
    ).div(
        1e6
    )  # TWh
    # .sort_values(ascending=True)
)

# Format output
balance = balance[
    abs(balance.values) > clip
].to_frame(  # Filter for entries > clipped value
    "Supply (+), Demand (-) [TWh]"
)
balance.style.format("{:,.2f}")  # Make style a bit prettier

We can also visualize this for a better overview:

In [ ]:
balance.plot.barh(
    figsize=(5, 12),
    xlabel="TWh",
    title=f"Electricity Energy Balance {scenario} (clipped at {clip} TWh)",
);

### Reminder: Visualizing results using `PyPSA.statistics`

As we already introduced in a previous workshop, it is actually much quicker to explore the data with plots generated using the `PyPSA.statistics` module.

For example the electricity energy balance we investigated before...

In [ ]:
fig, ax, _ = s2030.energy_balance.plot.bar(
    bus_carrier=["AC", "low voltage"],
    query=f"abs(value)>{clip*1e6}",
    height=6,
)
ax.set_title(f"Electricity Energy Balance {scenario} (Clipped at {clip} TWh)");

...or we can even explore time series interactively:

In [ ]:
s2030.energy_balance.iplot.area(
    bus_carrier=["AC", "low voltage"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    width=1800,
    height=600,
    query="snapshot < '2009-03'",
    title="Electricity Energy Balance NT 2030, January - February",
)

We can easily have a closer look at the production of a specific technology and a specific country. For example January wind production in the Netherlands in NT 2030.

In [ ]:
fig = s2030.energy_balance.iplot.area(
    facet_col="country",
    y="value",
    x="snapshot",
    carrier="wind",
    color="carrier",
    query="country == 'NL' and snapshot < '2009-02'",
    width=1800,
    height=500,
    title="Wind Production Netherlands NT 2030, January",
)

fig.update_layout(yaxis_title="Wind Production [MWh_el]")

As you can see, in January the Netherland's wind mix is largely dominated by Offshore Wind production.

### Compare results with ``PyPSA.NetworkCollections``

Lastly, one might also want to compare results between runs or planning years. To make this a bit easier, PyPSA v1.0 introduced a new object called `NetworkCollections`.

Let's have a look at how this is used in practice. First we define a network collection (`nc`) with our previously imported result networks for NT 2030 and NT 2040.

In [ ]:
nc = pypsa.NetworkCollection(networks)
nc

As we can see, our `NetworkCollection` contains two networks, the `NT 2030` network and the `NT 2040` network.

We can now use `PyPSA.statistics` accessor directly on this `NetworkCollection` instead of a single network to get the metrics for them simultaneously. 

Let's start again by defining a helper variable `sc` for the statistics accessor to make our life a bit easier going forward.

In [ ]:
sc = nc.statistics

Now, let's have a look at the electricity energy balance again but for both networks at once:

In [ ]:
ebs = (
    sc.energy_balance(
        groupby=["bus_carrier", "carrier"],
        aggregate_across_components=True,
    )
    .div(1e6)
    .unstack("network")
    .loc[["Electricity", "Electricity Low Voltage"]]
    .droplevel("bus_carrier")
)

# Format output
ebs = ebs[abs(ebs["NT 2030"]) > clip]  # Filter for entries > 1 TWh
ebs.style.format("{:,.2f}")  # Make style a bit prettier

We can again create a simple plot to make this a bit easier to investigate

In [ ]:
ebs.plot.barh(
    figsize=(6, 10),
    xlabel="Supply (+), Demand (-) [TWh]",
    title=f"Electricity Energy Balance NT (clipped at {clip} TWh)",
);

Similarly, we can also look into electricity prices in the system. 

Note that you can easily decide if you want to calculate the price weighted by load instead using `weighting='load'`.

In [ ]:
prices = sc.prices(bus_carrier="AC", groupby=False, weighting="time").unstack("network")
prices.head(10)

Again we plot them for easier readability

In [ ]:
prices.plot.bar(
    figsize=(25, 4),
    edgecolor="white",
    ylabel="€/MWh",
    xlabel="Bus Carrier",
    title="Electricity Price by node",
);

:::{Note}

Please keep in mind that the methodology used to implement hydrogen and electricity market coupling slightly differs from the TYNDP 2024 approach. Unlike the Market Model, which assumes a fixed hydrogen fuel price for hydrogen-to-power generation, Open-TYNDP couples electricity and hydrogen markets by using endogenous hydrogen fuel price for them. This results in high price spikes induced by load shedding in the coupled market. This is especially visible in the NT 2040 outcomes. Detailed electricity price benchmarks excluding load shedding are available on [Zenodo](https://zenodo.org/records/20303009).

:::

Of course `NetworkCollections` also work with `PyPSA.statistics` quick plotting API:

In [ ]:
fig = sc.prices.iplot.bar(
    bus_carrier=["AC"],
    x="name",
    y="value",
    color="network",
    stacked=True,
    width=1800,
    height=500,
    nice_names=False,
    title="Electricity Price by node",
)

fig.update_layout(yaxis_title="Prices [EUR/MWh]")

## Task 1: Grow comfortable with ``PyPSA.statistics``
Familiarize yourself with the statistics module (again) and explore the latest outcomes of Open-TYNDP using the different methods and plots introduced today.

**Hint**: You can also refer to the [introduction](#reminder-the-pypsa-statistics-module) above for more information on the different methods and parameters of ``PyPSA.statistics``.

## Task 2: Reproduce Benchmarks
(a) If you feel comfortable using `PyPSA.statistics`, you can try to reproduce the Open-TYNDP outcomes from the following example of our latest [benchmarking figures](https://zenodo.org/records/20303009).

Try it without looking at the previous example first.

In [ ]:
show_benchmarks(
    "benchmark_hydrogen_price_cy2009",
    [2030],
    "data/results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

(b) Optional: Try to exclude load shedding from the hydrogen price in 2040.

## Task 3 (Advanced): Inspect Outputs

(a) Can you verify the amount of wind generated on the Bornholm Energy Island in 2040 at *10.8 TWh*?

**Hint:** The Offshore Hub bus carrier is `AC_OH` and Borholm Energy Island bus is called `BEIOH01`.

(b) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

**Hint:** Look for `H2 pipeline` in the energy balance.

(c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?

## Interactive Exploration with PyPSA-Explorer

As we introduced in the last workshop, PyPSA-Explorer is an interactive dashboard for visualizing and analyzing energy system networks. It provides:
- Energy balance analysis with both time series and aggregated views
- Capacity planning visualizations by technology and region
- Economic analysis showing CAPEX/OPEX breakdowns
- Interactive geographical network maps
- Support for visualising multiple networks

### Reminder: Using the Dashboard

Once the dashboard opens, you can explore these key features:

**1. Energy Balance Tab**
   - View production, consumption, and storage patterns over time
   - Switch between time series and aggregated views
   - Filter by energy carrier (electricity, hydrogen, etc.)
   - Filter by country or region

**2. Capacity Tab**
   - Analyze installed capacities across scenarios
   - Compare capacity buildout between 2030 and 2040
   - View breakdowns by technology type and region

**3. Economics Tab**
   - Examine costs and revenues
   - Review CAPEX and OPEX breakdowns by technology
   - Compare regional cost distributions
   - Assess investment requirements

**4. Network Map**
   - Visualize the geographical network layout
   - View an interactive map with network components
   - Zoom and pan to explore specific regions

**Tip:** Use the scenario selector buttons in the top-right corner to switch between NT 2030 and NT 2040 scenarios.

:::{note}

 PyPSA-Explorer can be launched in different ways depending on your environment:

- **Local Jupyter**: Use the terminal command (recommended) or inline display
- **Google Colab**: The dashboard launches inline, embedded directly in the notebook

Follow the instructions below for your specific environment.

:::

In [ ]:
# Detect if running on Google Colab
try:
    from google.colab import output

    IN_COLAB = True
    print(f"This notebook is running on Google Colab!")
except ImportError:
    IN_COLAB = False
    print(f"This notebook is running locally !")

port = 8050

#### For Local Users

If you're running locally, we **recommend** launching PyPSA-Explorer from the terminal for optimal performance:

```bash
pypsa-explorer data/results-0.7.1/NT-cy2009-20260520/networks/base_s_all___2030.nc:NT_2030 data/results-0.7.1/NT-cy2009-20260520/networks/base_s_all___2040.nc:NT_2040
```

This command opens the dashboard in your default browser at http://localhost:8050.

**Alternative**: The cell below can launch the dashboard inline within the notebook, though the terminal method provides better performance and responsiveness.

In [ ]:
# Terminal method recommended
USE_TERMINAL = True  # Change to False if you want to launch from the notebook instead

if not IN_COLAB and not USE_TERMINAL:
    # Local Jupyter: Inline display
    app = create_app(networks)
    app.run(jupyter_mode="tab", port=port, debug=False)

#### For Google Colab Users

Running PyPSA-Explorer on Google Colab requires a small workaround to display the dashboard properly inside the notebook.

We already imported a useful helper function we introduced in the last workshop to handle this: `run_pypsa_explorer_in_colab()`

In [ ]:
if IN_COLAB:
    run_pypsa_explorer_in_colab(networks, port)

**Tip for Colab users:** To view the dashboard in fullscreen mode, click the three dots (⋮) in the top-right corner of the output cell and select **"View output fullscreen"**.

## Task 4: Navigate ``PyPSA-Explorer``
Let's look at the latest Open-TYNDP NT scenario results (v0.7.1) again and explore them interactively using the PyPSA-Explorer. Try again to find some specific insights we manually extracted in the previous section.

(a) Can you verify the amount of wind generated on the Bornholm Energy Island in 2040 at *10.8 TWh*?

(a) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

(c) Can you verify the observed correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040?

# Cost-Benefit Analysis

## Workflow management using `Snakemake` and `pixi`

- Talk about how the Open-TYNDP project utilizes `Snakemake` as a workflow management system
- We also use `pixi` to manage environments and to enable ease-of-use of `Snakemake`

### Reminder: `Snakemake`

<img src="https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/792b8474ab5096e5ab8db2822af4fcd9fe659eb6/open-tyndp-workshops/snakemake_logo.png" width="500px" />

The `Snakemake` workflow management system is a tool to create reproducible and scalable data analyses.
Workflows are described via a human readable, Python based language. They can be seamlessly scaled to server, cluster, grid, and cloud environments, without the need to modify the workflow definition.

Snakemake follows the [GNU Make](https://www.gnu.org/software/make) paradigm: workflows are defined in terms of so-called `rules` that specify how to create a set of output files from a set of input files. Dependencies between the rules are determined automatically, creating a DAG (directed acyclic graph) of jobs that can be automatically parallelized.

:::{note}
Documentation for this package is available at https://snakemake.readthedocs.io/. You can also check out a [slide deck Snakemake Tutorial](https://slides.com/johanneskoester/snakemake-tutorial) by Johannes Köster (2024).

Mölder, F., Jablonski, K.P., Letcher, B., Hall, M.B., Tomkins-Tinch, C.H., Sochat, V., Forster, J., Lee, S., Twardziok, S.O., Kanitz, A., Wilm, A., Holtgrewe, M., Rahmann, S., Nahnsen, S., Köster, J., 2021. Sustainable data analysis with Snakemake. F1000Res 10, 33.

Add note to refer to previous workshop
:::


#### Using `snakemake`

You can trigger the workflow by specifying a target file, like `data/benchmark.pdf`, or any intermediate file:
```bash
snakemake -call data/benchmark.pdf
```

Alternatively, you can also execute the workflow by calling a rule that produces an intermediate file:
```bash
snakemake -call build_data
```
**NOTE:** You cannot call a rule that includes a wildcard without specifying what the wildcard should be filled with. Otherwise, Snakemake will not know what to propagate back.

Or you can call the common rule `all` which can be used to execute the entire workflow. It takes the final workflow output as its input and thus requires all previous dependent rules to be run as well:
```bash
snakemake -call all
```

Because we defined the `all` rule as first in the `Snakefile`, this rule is assumed to be the default and the following also works:
```bash
snakemake -call
```

A very important feature is the `-n` flag which executes a `dry-run`. It is recommended to always first execute a `dry-run` before the actual execution of a workflow. This simply prints out the DAG of the workflow to investigate without actually executing it.

Let's try this out and investigate the output:

In [ ]:
! snakemake -call -n

### Introducing: `pixi`

<img src="https://raw.githubusercontent.com/prefix-dev/pixi/9f94b8a24353ca88d21846b72438c6066e8adc74/docs/assets/pixi-logo.svg" width="150px" />



`pixi` is a cross-platform, multi-language package management and workflow tool. It is built on the foundation of the `conda` ecosystem.

:::{note}
Documentation for this package is available at https://pixi.prefix.dev/latest/.

:::


#### Using `pixi`

- we are using `pixi` for package/environment management
- we are able to wrap snakemake commands into pixi commands
- for remainder of notebook, we will use pixi

Give examples of how we wrapped specific snakemake commands into pixi commands, to allow us to run shorter commands

### Coupled vs decoupled SB-CBA workflow

<center>

![](cba_multi_proj.png)
![](cba_from_presolved.png)

</center>

Note:
- Connection between SB solve and CBA
- Mention that there are many many rules before solve_sector_network_myopic - we just chose to highlight the handoff between SB and CBA here
- Number of projects - only 2 shown here but for each project, the DAG expands and blows up

Note:
- With this there are no rules before the retrieve rule - we start from this point
- But all else is the same

## Exploring Open-TYNDP CBA

In [ ]:
# Uncomment the next line for running this notebook on Colab
# !wget -qO- https://pixi.sh/install.sh | sh

### Cloning the Open-TYNDP project

First, navigate into the `data/` folder:

In [ ]:
# TODO: create data/ folder if it doesn't exist, and cd into it before cloning the repo
os.chdir("data/")
print("Directory changed to:", os.getcwd())

Clone the Open-TYNDP repository directly:

In [ ]:
if os.path.basename(os.getcwd()) == "data":
    if not os.path.exists("open-tyndp"):
        ! git clone https://github.com/open-energy-transition/open-tyndp.git
    else:
        print("open-tyndp directory already exists, skipping clone")
else:
    print("Warning: Not in data/ directory. Current directory:", os.getcwd())

Now, navigate into the the `open-tyndp` directory:

In [ ]:
os.chdir("open-tyndp/")
print("Directory changed to:", os.getcwd())

In [ ]:
! git checkout cba-low-res

(add instruction to install pixi - copy from workshop 3)

Use `pixi` to install the `open-tyndp` environment:

In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]

In [ ]:
! pixi install -e open-tyndp

### Running a coupled SB->CBA workflow

To run the CBA workflow, the `pixi run tyndp-cba` runs the full process: taking a solved SB network -> performing the TOOT/PINT analysis on a project -> calculating the CBA indicators.

The default Open-TYNDP settings can be found in the `config/config.tyndp.yaml` file.

TODO: add chunks of config file (refer to previous workshop about manual adjustments)
- add snippets of default values (runs, horizons, snapshots, projects, etc)

mention:
- because of notebook environment, we are focusing on command line changes, but in reality it's easier to modify in config files directly using dedicated IDE

First, let's check what running the full could look like by doing a dry-run of `pixi run tyndp-cba`:

In [ ]:
! pixi run tyndp-cba -n

We can specify the specific scenario (e.g, `NT`), the project (e.g., `t4`), and planning horizon (e.g., 2030) we want to run also directly in the command line.

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Notice how the `count` of steps changed when we specified a single run, project, and horizon.
Comparatively, the default settings run the full list of scenarios we have, 5 projects (t4, t16, t28, t33, t35), and two planning horizons (2030 and 2040). So, the full default workflow will run many more steps.

#### Checkpoints

You may notice above that the number of rules is actually quite low.

There is a `checkpoint` in the workflow, called `clean_projects`, that first checks how many projects are being asked to run before building out the full DAG.
It tells the workflow which CBA projects exist, which project IDs to run, and which method applies (TOOT/PINT).

Before the `clean_projects` step runs, Snakemake does not know the full list of project jobs it needs to create.
The step downloads and cleans the external CBA project database, tells the workflow the full list of projects available, which projects the user wants to evaluate, and how each should be evaluated.
After `clean_projects` finishes, Snakemake can read the cleaned CSV and expand the DAG into concrete jobs.

Thus, what we should do first here is run the workflow just up until the checkpoint, then only after that can we run the full workflow again -- in which case, the DAG would show the actual number of jobs that would be run.

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

Now that the checkpoint is complete, we can re-check the DAG of how many steps are needed to run the CBA workflow.

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

#### Task X: Change the configuration directly in the `config/config.tyndp.yaml`

- Change the config settings to match above
  - Change scenario name in config.tyndp.yaml
  - Change parameters within NT in scenarios.tyndp.yaml
- Come back here
- Run simplified command line

In [ ]:
# Should give the same output as above, without the arguments
! pixi run tyndp-cba -n

### Running different and multiple climate years

(Add some info here about how different climate scenarios are defined? Navigate into `config/scenarios.yaml`?)

TODO:
- add snippets of yaml here to show as examples
- can also show live
- stress that in practice it's easier to modify in yaml

First, we again need to run up to the checkpoint for the different climate years:

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["NT-cy2008", "NT-cy2009", "NT-cy1995"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

So, we can run for just another climate year, such as the 1995 climate year for the NT scenario (`NT-cy2008`):

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cy2008"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Or, we can run all climate years (1995, 2008, and 2009) for the `NT` scenario:

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cyears"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Note how the total count of steps is much higher (especially for CBA-specific steps such as `make_indicators` and weather-dependent steps such as `build_renewable_profiles_pecd`) when multiple climate years are run.

#### Task X: Create your own climate year and run it

Go into `scenarios.tyndp.yaml` to create your own climate year run.

Hints:
1. Copy an existing climate year run (e.g., `NT-cy2008`) and modify the details to your liking. Change the name.
2. You would need to first run the new climate year up until the checkpoint.
3. After the checkpoint, you can use `pixi run tyndp-cba` to run the full workflow for your new climate year.

Warning:
We used previous set of PECD data. So we can only run 1995, 2008, and 2009 at the moment.

Mention the benefit: if we have more climate years in the year, this is how we would add it.

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["frog"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["frog"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

### Running a decoupled SB->CBA Workflow

Based on the DAGs shown in our dry runs, you can see that there are quite a lot of steps and rules that need to be run. A lot of these rules are specific to the SB stage of Open-TYNDP.

We have the flexibility to bypass running and solving the SB workflow each time, by instead downloading a pre-solved SB network that we have released. By doing this, when running the CBA workflow, we start the process from an already solved SB network, reducing the number of steps significantly.

This can be easily done by setting the `cba.use_presolved` setting to `true`. By default, this downloads and uses the latest release we have (at the time of this workshop, that is v0.7.1).

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["frog"]}' 'cba={"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"planning_horizons":[2030],"projects":["t4"]}' -n

Note how the number of steps has decreased down from over 100 to just ~37, and we see some new steps not previously seen before, such as `retrieve_presolved_sb_networks`.

#### Task (optional): Running and solving a CBA project with a pre-solved network

- Checkout to workshop branch
- Mention warning that it takes 15 minutes to solve
- Let people run it during the break

# Solutions

## Task 2: Reproduce benchmarks

In [ ]:
show_benchmarks(
    "benchmark_hydrogen_price_cy2009",
    [2030],
    "data/results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

In [ ]:
# (a) Try to reproduce the Open-TYNDP outcomes from the hydrogen prices example above from our latest [benchmarking figures](https://zenodo.org/records/20303009).
prices = (
    s2030.prices(bus_carrier="H2", weighting="time").to_frame("Open-TYNDP").sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_H2", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.1f} EUR/MWh_H2)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Hydrogen Price - Scenario NT - CY 2009 - Year 2030")
ax.tick_params(axis="x", rotation=45)

In [ ]:
s2040 = networks["NT 2040"].stats

In [ ]:
# (b) Optional: Try to exclude load shedding from the hydrogen price in 2040.
voll = 3000  # EUR/ MWh_H2
prices = (
    s2040.prices(
        bus_carrier="H2",
        weighting="time",
        groupby_time=False,
    )
    .pipe(lambda x: x.where(x < voll * 0.98))  # Add 2% of numerical tolerance
    .mean(axis=1)
    .to_frame("value")
    .sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_H2", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.1f} EUR/MWh_H2)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Hydrogen Price excl. Load Shedding - Scenario NT - CY 2009 - Year 2040")
ax.tick_params(axis="x", rotation=45)

## Task 3: Investigate outputs

In [ ]:
# (a) Can you verify the amount of wind generated on the Bornholm Energy Island in 2040 at *10.8 TWh*?
(
    s2040.energy_balance(
        bus_carrier="AC_OH",
        aggregate_across_components=True,
        groupby=["bus", "carrier"],
        components="Generator",
    )
    .div(1e6)  # TWh
    .to_frame("Wind Generation [TWh]")
    .loc["BEIOH01"]
)

In [ ]:
# (b) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

(
    s2040.energy_balance(
        bus_carrier="H2",
        carrier="H2 pipeline",
        aggregate_across_components=True,
        groupby=["carrier", "bus"],
    )
    .div(1e6)
    .sort_values(ascending=False)
    .to_frame("H2 Import (+), H2 Export (-) [TWh]")
    .head()
    .style.format("{:,.2f}")  # Make style a bit prettier
)

In [ ]:
# (c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?
s2040.energy_balance.iplot.area(
    bus_carrier=["AC", "H2"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    facet_row="bus_carrier",
    facet_col="country",
    sharex=False,
    sharey=False,
    query="snapshot < '2009-06-08' and snapshot >= '2009-06-01' and country == 'DE'",
    width=1800,
    height=500,
)

We can observe for night of the 6th of June, that wind production drops along with solar electricity production resulting in no hydrogen production via electrolysis for that time. Instead, German hydrogen demand is met via H2 pipeline imports as well as from Cavern Storages and blue Hydrogen production. Pumped hydro, battery storage and electricity imports are utilized to support electricity production in the same period to meet the remaining exogenous electricity demand.

# Notebook clean up

In [ ]:
# Only clean up data when running in CI environment
if os.getenv("CI"):
    rm_dir = "data/results-0.7.1"
    print(
        f"CI environment detected. Cleaning up notebook data by removing '{rm_dir}' and '{rm_dir}.zip'."
    )
    shutil.rmtree(rm_dir, ignore_errors=True)
    Path(f"{rm_dir}.zip").unlink(missing_ok=True)
else:
    print("Skipping cleanup (not in CI environment).")